# Self-Contrastive Decoding

**Idee:** Maximiere die Diversität relativ zu bereits generierten Samples

### Vergleich mit Credal Decoding

| | Credal Decoding | Self-Contrastive |
|---|---|---|
| Ziel | Annäherung an Human Credal Set | Maximierung interner Diversität |
| Braucht Human-Daten? | Ja | Nein |
| Feedback-Signal | Distanz zum Human-Centroid | Diversität der bisherigen Samples |
| τ-Steuerung | höher wenn zu homogen vs. Human | höher wenn Samples zu ähnlich zueinander |

### Drei Strategien

1. **Self-Contrastive (Diversity Maximizer):** τ steigt wenn Samples zu ähnlich, sinkt wenn zu verschieden
2. **Entropy-Guided:** τ folgt der Token-Entropie des Modells pro Prompt
3. **Strategy Switching:** Wechsle zwischen Decoding-Strategien pro Sample

In [2]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time, json, os
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from sentence_transformers import SentenceTransformer
import spacy
from scipy.spatial.distance import cdist
from scipy.stats import wasserstein_distance
from itertools import combinations
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

/Users/theresawinner/Desktop/Uni/Master/Masterarbeit/Masterthesis/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [ ]:
# Konfiguration
N_PROMPTS = 10
N_SAMPLES = 10
MAX_NEW_TOKENS = 150

MODEL_IDS = {
    'qwen-3-4b': 'Qwen/Qwen3-4B',
}

BASELINES = {
    'temp_0.7':  dict(temperature=0.7, top_p=0.95, top_k=0),
    'temp_1.2':  dict(temperature=1.2, top_p=0.95, top_k=0),
    'top_p_0.9': dict(temperature=1.0, top_p=0.90, top_k=0),
    'top_k_40':  dict(temperature=1.0, top_p=1.0,  top_k=40),
    'typical':   dict(temperature=0.95, top_p=0.92, top_k=0),
}

OUT_DIR = Path('results_sc')
OUT_DIR.mkdir(exist_ok=True)
print(f'Experiment: {N_PROMPTS} Prompts × {N_SAMPLES} Samples')

Experiment: 10 Prompts × 10 Samples


## 2. Daten laden (identisch zum Credal-Experiment)

In [4]:
from datasets import load_dataset
from collections import defaultdict

def load_prompts(n_prompts=10, n_cont=10):
    print('Lade WritingPrompts...')
    ds = load_dataset('euclaise/writingprompts', split='train')
    prompt_stories = defaultdict(list)
    for item in ds:
        p, s = item['prompt'].strip(), item['story'].strip()
        if 20 <= len(p) <= 500 and 50 <= len(s.split()) <= 1000:
            prompt_stories[p].append(s)
    valid = [(p, list(set(ss))[:n_cont]) for p, ss in prompt_stories.items() if len(set(ss)) >= n_cont]
    valid.sort(key=lambda x: np.std([len(s.split()) for s in x[1]]), reverse=True)
    sel = valid[:n_prompts]
    print(f'Ausgewählt: {len(sel)} Prompts')
    return [p for p, _ in sel], {i: ss for i, (_, ss) in enumerate(sel)}

PROMPTS, HUMAN_STORIES = load_prompts(N_PROMPTS, N_SAMPLES)

Lade WritingPrompts...


Ausgewählt: 10 Prompts


In [ ]:
# SBERT + spaCy
sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device=DEVICE)
nlp = spacy.load('en_core_web_sm')

def compute_diversity(texts):
    if len(texts) < 2: return np.array([0., 0., 0.])
    pairs = list(combinations(range(len(texts)), 2))
    embs = sbert.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    d_sem = np.mean([1 - np.dot(embs[i], embs[j]) for i, j in pairs])
    bgs = [set(zip(t.lower().split()[:-1], t.lower().split()[1:])) for t in texts]
    d_lex = np.mean([1 - len(bgs[i]&bgs[j])/max(len(bgs[i]|bgs[j]),1) for i,j in pairs])
    def pt(t):
        tags = [tk.pos_ for tk in nlp(t[:800])]
        return set(zip(tags[:-2], tags[1:-1], tags[2:])) if len(tags)>=3 else set()
    ps = [pt(t) for t in texts]
    d_syn = np.mean([1 - len(ps[i]&ps[j])/max(len(ps[i]|ps[j]),1) for i,j in pairs])
    return np.array([d_sem, d_lex, d_syn])

print('Metriken bereit.')

In [ ]:
# Human-Vektoren berechnen
print('Berechne Human-Diversität:')
human_vectors = np.array([compute_diversity(HUMAN_STORIES[i][:N_SAMPLES]) for i in range(len(PROMPTS))])
H_CENTROID = human_vectors.mean(0)
H_STDS = human_vectors.std(0)

print(f'Human Centroid: {H_CENTROID.round(3)}')

## 3. Modell laden

In [ ]:
def load_model(model_id):
    print(f'Lade {model_id}...')
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant, device_map='auto')
    mdl.eval()

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        mdl.config.pad_token_id = tok.eos_token_id

    print(f'  Geladen: {sum(p.numel() for p in mdl.parameters())/1e9:.1f}B params')
    return mdl, tok

def format_prompt(prompt, tokenizer):
    sys = 'You are a creative fiction writer. Write a story continuation. Output only the story text.'

    if hasattr(tokenizer, 'apply_chat_template') and tokenizer.chat_template:
        try:
            return tokenizer.apply_chat_template(
                [{'role':'system','content':sys}, {'role':'user','content':prompt}],
                tokenize=False, add_generation_prompt=True)
        except: pass

    return f'{sys}\n\n{prompt}\n\n'

def generate(prompt, model, tokenizer, n=10, temperature=0.7, top_p=0.95, top_k=0):
    fmt = format_prompt(prompt, tokenizer)
    inp = tokenizer(fmt, return_tensors='pt').to(model.device)
    plen = inp['input_ids'].shape[1]

    kw = dict(**inp, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
              temperature=max(temperature, 0.01), num_return_sequences=n,
              pad_token_id=tokenizer.eos_token_id)
    
    if top_k > 0: kw['top_k'] = top_k
    else: kw['top_p'] = top_p

    with torch.no_grad():
        out = model.generate(**kw)
        
    return [tokenizer.decode(s[plen:], skip_special_tokens=True).strip() for s in out if len(s) > plen]

## 4. Metriken

In [ ]:
DIM = ['Semantic', 'Lexical', 'Syntactic']

def mean_nn(a, b):
    return 0.5 * (cdist(a,b).min(1).mean() + cdist(b,a).min(1).mean())

def w_per_dim(a, b):
    return np.array([wasserstein_distance(a[:,i], b[:,i]) for i in range(3)])

def evaluate(vecs, ref):
    w = w_per_dim(vecs, ref)
    return dict(
        mean_nn=mean_nn(vecs, ref),
        w_total=float(w.sum()),
        centroid_d=float(np.linalg.norm(vecs.mean(0) - ref.mean(0))),
        wasserstein=w,
        mean=vecs.mean(0),
    )

## 5. Unsupervised Decoding-Strategien

In [ ]:
# ============================================================
# Strategie A: Self-Contrastive Decoding
# ============================================================
# Idee: Berechne die Diversität der bisherigen Samples.
# Wenn sie UNTER einem Zielwert liegt -> tau rauf.
# Der Zielwert wird nicht extern gesetzt, sondern als
# "maximale bisher beobachtete Diversität" adaptiv bestimmt.

def self_contrastive_decode(
    prompts, model, tokenizer,
    tau_init=0.7, alpha=0.12, tau_min=0.3, tau_max=2.5,
    target_quantile=0.75,  # Ziel: oberes Quartil der bisherigen Diversität
):
    """
    Self-Contrastive: Maximiere Diversität ohne externes Ziel.
    
    Feedback-Signal: Liegt die aktuelle Diversität unter dem
    target_quantile der bisherigen Diversitäten? -> tau rauf.
    """
    tau = tau_init
    vecs, history = [], []
    
    for pi, prompt in enumerate(prompts):
        set_seed(pi * 100)
        texts = generate(prompt, model, tokenizer, n=N_SAMPLES, temperature=tau)
        v = compute_diversity(texts)
        vecs.append(v)
        
        # Gesamtdiversität als Skalar (Mittelwert der 3 Dimensionen)
        div_score = v.mean()
        
        history.append(dict(step=pi, tau=tau, div_score=div_score, vector=v.copy()))
        
        if len(vecs) >= 3:
            all_scores = [vv.mean() for vv in vecs]
            target = np.quantile(all_scores, target_quantile)
            
            # Wenn unter Ziel: zu homogen -> tau rauf
            delta = target - div_score
            tau = np.clip(tau + alpha * delta / (np.std(all_scores) + 1e-6), tau_min, tau_max)
        
        if (pi+1) % 5 == 0:
            print(f'  [{pi+1}/{len(prompts)}] τ={tau:.2f} div={div_score:.3f}', end='\r')
    
    print()
    return np.array(vecs), history


# ============================================================
# Strategie B: Entropy-Guided Decoding
# ============================================================
# Idee: Miss die Entropie der Output-Distribution des Modells
# für den gegebenen Prompt. Hohe Entropie = Modell ist unsicher
# = der Prompt ist ambig = höhere tau ist angemessen.

def compute_prompt_entropy(prompt, model, tokenizer, n_tokens=20):
    """Mittlere Token-Entropie für die ersten n_tokens der Generierung"""
    fmt = format_prompt(prompt, tokenizer)
    inp = tokenizer(fmt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inp, max_new_tokens=n_tokens, do_sample=False,
            return_dict_in_generate=True, output_scores=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    if not out.scores:
        return 1.0  # fallback
    
    entropies = []
    for logits in out.scores:
        probs = F.softmax(logits[0], dim=-1)
        ent = -(probs * torch.log(probs + 1e-10)).sum().item()
        entropies.append(ent)
    
    return np.mean(entropies)


def entropy_guided_decode(
    prompts, model, tokenizer,
    tau_base=0.7, tau_scale=0.15, tau_min=0.3, tau_max=2.5,
):
    """
    Entropy-Guided: tau proportional zur Prompt-Entropie.
    
    Hohe Entropie -> Modell ist unsicher -> mehr Variation erlauben.
    Niedrige Entropie -> klare Antwort -> weniger Variation.
    """
    vecs, history = [], []
    entropies_so_far = []
    
    for pi, prompt in enumerate(prompts):
        # Berechne Prompt-Entropie
        ent = compute_prompt_entropy(prompt, model, tokenizer)
        entropies_so_far.append(ent)
        
        # Normalisiere relativ zu bisherigen Prompts
        if len(entropies_so_far) >= 2:
            ent_z = (ent - np.mean(entropies_so_far)) / (np.std(entropies_so_far) + 1e-6)
        else:
            ent_z = 0.0
        
        tau = np.clip(tau_base + tau_scale * ent_z, tau_min, tau_max)
        
        set_seed(pi * 100)
        texts = generate(prompt, model, tokenizer, n=N_SAMPLES, temperature=tau)
        v = compute_diversity(texts)
        vecs.append(v)
        
        history.append(dict(step=pi, tau=tau, entropy=ent, vector=v.copy()))
        
        if (pi+1) % 5 == 0:
            print(f'  [{pi+1}/{len(prompts)}] τ={tau:.2f} H={ent:.1f}', end='\r')
    
    print()
    return np.array(vecs), history


# ============================================================
# Strategie C: Strategy Switching
# ============================================================
# Idee: Anstatt eine Strategie zu adaptieren, wechsle
# zwischen verschiedenen Strategien pro Sample.
# Die natürliche Variation zwischen Strategien erzeugt Diversität.

SWITCH_POOL = [
    dict(temperature=0.5, top_p=0.95, top_k=0),
    dict(temperature=0.8, top_p=0.95, top_k=0),
    dict(temperature=1.2, top_p=0.95, top_k=0),
    dict(temperature=1.0, top_p=0.85, top_k=0),
    dict(temperature=1.0, top_p=1.0,  top_k=30),
]

def strategy_switching_decode(prompts, model, tokenizer):
    """
    Strategy Switching: Generiere jedes Sample mit einer
    anderen Strategie aus dem Pool. Die Mischung erzeugt
    natürliche Diversität.
    """
    vecs, history = [], []
    
    for pi, prompt in enumerate(prompts):
        
        # Generiere Samples mit wechselnden Strategien
        all_texts = []
        for si in range(N_SAMPLES):
            strat = SWITCH_POOL[si % len(SWITCH_POOL)]
            set_seed(pi * 100 + si)
            texts = generate(prompt, model, tokenizer, n=1, **strat)
            all_texts.extend(texts)
        
        v = compute_diversity(all_texts[:N_SAMPLES])
        vecs.append(v)
        history.append(dict(step=pi, vector=v.copy()))
        
        if (pi+1) % 5 == 0:
            print(f'  [{pi+1}/{len(prompts)}]', end='\r')
    
    print()
    return np.array(vecs), history


print('Strategien definiert.')

## 6. Experiment durchführen

In [ ]:
ALL = {}  # model -> strategy -> results

for mname, mid in MODEL_IDS.items():
    print(f'\n{"="*60}')
    print(f'MODELL: {mname}')
    print(f'{"="*60}')
    mdl, tok = load_model(mid)
    ALL[mname] = {}
    
    # --- Baselines ---
    for sname, params in BASELINES.items():
        print(f'\n  Baseline: {sname}')
        t0 = time.time()
        vl = []

        for pi, prompt in enumerate(PROMPTS):
            set_seed(pi * 100)
            texts = generate(prompt, mdl, tok, n=N_SAMPLES, **params)
            vl.append(compute_diversity(texts))
            if (pi+1)%5==0: print(f'    [{pi+1}/{len(PROMPTS)}]', end='\r')

        print()
        vecs = np.array(vl)
        ALL[mname][sname] = dict(vectors=vecs, **evaluate(vecs, human_vectors))
        
        print(f'    Done ({time.time()-t0:.0f}s) — centroid_d={ALL[mname][sname]["centroid_d"]:.3f}')
    
    # --- Self-Contrastive ---
    print(f'\n  Self-Contrastive:')
    t0 = time.time()
    v, h = self_contrastive_decode(PROMPTS, mdl, tok)
    ALL[mname]['self_contrastive'] = dict(vectors=v, history=h, **evaluate(v, human_vectors))
    print(f'    Done ({time.time()-t0:.0f}s) — centroid_d={ALL[mname]["self_contrastive"]["centroid_d"]:.3f}')
    
    # --- Entropy-Guided ---
    print(f'\n  Entropy-Guided:')
    t0 = time.time()
    v, h = entropy_guided_decode(PROMPTS, mdl, tok)
    ALL[mname]['entropy_guided'] = dict(vectors=v, history=h, **evaluate(v, human_vectors))
    print(f'    Done ({time.time()-t0:.0f}s) — centroid_d={ALL[mname]["entropy_guided"]["centroid_d"]:.3f}')
    
    # --- Strategy Switching ---
    print(f'\n  Strategy Switching:')
    t0 = time.time()
    v, h = strategy_switching_decode(PROMPTS, mdl, tok)
    ALL[mname]['switching'] = dict(vectors=v, history=h, **evaluate(v, human_vectors))
    print(f'    Done ({time.time()-t0:.0f}s) — centroid_d={ALL[mname]["switching"]["centroid_d"]:.3f}')
    
    del mdl, tok
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('  Modell entladen.')

## 7. Lade Credal-Ergebnisse zum Vergleich

In [ ]:
# Versuche Credal-Ergebnisse aus dem vorherigen Experiment zu laden
credal_loaded = False
credal_path = Path('results/results.json')
with open(credal_path) as f:
    credal_data = json.load(f)
for mname in ALL:
    if mname in credal_data and 'credal' in credal_data[mname]:
        cd = credal_data[mname]['credal']
        vecs = np.array(cd['vectors'])
        ALL[mname]['credal'] = dict(vectors=vecs, **evaluate(vecs, human_vectors))
        credal_loaded = True
        print(f'Credal-Ergebnisse für {mname} geladen.')

## 8. Ergebnistabelle

In [ ]:
NEW_STRATS = ['self_contrastive', 'entropy_guided', 'switching']
if credal_loaded:
    NEW_STRATS = ['credal'] + NEW_STRATS

all_strats = list(BASELINES.keys()) + NEW_STRATS

print(f'{"Model":<12} {"Strategy":<18} {"Mean_NN":>8} {"W_total":>8} {"Centroid":>9} {"W_sem":>7} {"W_lex":>7} {"W_syn":>7}')
print('=' * 82)

for mname in ALL:
    for sname in all_strats:

        if sname not in ALL[mname]: continue
        r = ALL[mname][sname]
        w = r['wasserstein']
        is_new = sname in NEW_STRATS
        marker = '  ◄' if is_new else ''

        print(f'{mname:<12} {sname:<18} {r["mean_nn"]:>8.3f} {r["w_total"]:>8.3f} '
              f'{r["centroid_d"]:>9.3f} {w[0]:>7.3f} {w[1]:>7.3f} {w[2]:>7.3f}{marker}')
        
    print('-' * 82)

# Ranking
print(f'\n--- Ranking nach Centroid-Distanz (kleiner = besser) ---')
for mname in ALL:
    scores = [(s, ALL[mname][s]['centroid_d']) for s in all_strats if s in ALL[mname]]
    scores.sort(key=lambda x: x[1])
    
    print(f'\n{mname}:')
    for rank, (s, v) in enumerate(scores, 1):
        label = '(supervised)' if s == 'credal' else '(unsupervised)' if s in ['self_contrastive','entropy_guided','switching'] else '(baseline)'
        print(f'  {rank}. {s:<18} {v:.3f}  {label}')

## 9. Visualisierungen

In [ ]:
# Barplot: Alle Strategien

model_list = list(ALL.keys())
plot_strats = ['temp_0.7', 'temp_1.2', 'top_k_40']
if credal_loaded: plot_strats.append('credal')
plot_strats += ['self_contrastive', 'entropy_guided', 'switching']

colors_map = {
    'temp_0.7': '#ef4444', 'temp_1.2': '#f97316', 'top_p_0.9': '#eab308',
    'top_k_40': '#a855f7', 'typical': '#6366f1',
    'credal': '#059669',
    'self_contrastive': '#0ea5e9', 'entropy_guided': '#8b5cf6', 'switching': '#ec4899',
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(model_list))
w = 0.8 / len(plot_strats)

for i, s in enumerate(plot_strats):
    vals_cd = [ALL[m].get(s, {}).get('centroid_d', 0) for m in model_list]
    vals_wt = [ALL[m].get(s, {}).get('w_total', 0) for m in model_list]
    c = colors_map.get(s, '#888')
    ax1.bar(x + i*w, vals_cd, w*0.9, label=s, color=c, alpha=0.85)
    ax2.bar(x + i*w, vals_wt, w*0.9, label=s, color=c, alpha=0.85)

for ax, title, ylabel in [(ax1, 'Centroid-Distanz (↓)', 'Centroid Distance'),
                           (ax2, 'Wasserstein Total (↓)', 'W_total')]:
    ax.set_xticks(x + w*len(plot_strats)/2)
    ax.set_xticklabels([m.upper() for m in model_list])
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=7, ncol=2)

plt.suptitle('Supervised (Credal) vs. Unsupervised Strategien', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'comparison_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Credal Sets 2D: Alle neuen Strategien
from scipy.spatial import ConvexHull

strats_to_plot = NEW_STRATS
strat_colors = {'credal':'#059669', 'self_contrastive':'#0ea5e9',
                'entropy_guided':'#8b5cf6', 'switching':'#ec4899'}

for mname in ALL:
    n = len(strats_to_plot)
    fig, axes = plt.subplots(1, n, figsize=(5.5*n, 5))
    if n == 1: axes = [axes]
    
    for ax, sname in zip(axes, strats_to_plot):
        if sname not in ALL[mname]: continue
        col = strat_colors.get(sname, '#888')
        
        # Human
        ax.scatter(human_vectors[:,0], human_vectors[:,1], c='#2563eb', alpha=0.15, s=6)
        try:
            hull = ConvexHull(human_vectors[:,:2])
            for s in hull.simplices:
                ax.plot(human_vectors[s,0], human_vectors[s,1], c='#2563eb', alpha=0.4, lw=1)
        except: pass
        
        # Strategie
        sv = ALL[mname][sname]['vectors']
        ax.scatter(sv[:,0], sv[:,1], c=col, alpha=0.5, s=15)
        try:
            hull = ConvexHull(sv[:,:2])
            for s in hull.simplices:
                ax.plot(sv[s,0], sv[s,1], c=col, alpha=0.6, lw=1.5)
        except: pass
        
        cd = ALL[mname][sname]['centroid_d']
        ax.set_title(f'{sname}\n(cd={cd:.3f})', fontweight='bold')
        ax.set_xlabel('Semantic'); ax.set_ylabel('Lexical')
    
    plt.suptitle(f'{mname.upper()}: Unsupervised Strategien vs. Human', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'credal_2d_{mname}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# τ-Evolution: Self-Contrastive vs. Entropy-Guided

for mname in ALL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    
    if 'self_contrastive' in ALL[mname]:
        h = ALL[mname]['self_contrastive']['history']
        ax1.plot([x['step'] for x in h], [x['tau'] for x in h], 'o-', c='#0ea5e9', lw=2, ms=5)
        ax1b = ax1.twinx()
        ax1b.plot([x['step'] for x in h], [x['div_score'] for x in h], 's-', c='#f59e0b', lw=1.5, ms=4, alpha=0.7)
        ax1b.set_ylabel('Div Score', color='#f59e0b')
        ax1.set_ylabel('τ', color='#0ea5e9')
        ax1.set_xlabel('Prompt')
        ax1.set_title('Self-Contrastive', fontweight='bold')
    
    if 'entropy_guided' in ALL[mname]:
        h = ALL[mname]['entropy_guided']['history']
        ax2.plot([x['step'] for x in h], [x['tau'] for x in h], 'o-', c='#8b5cf6', lw=2, ms=5)
        ax2b = ax2.twinx()
        ax2b.plot([x['step'] for x in h], [x['entropy'] for x in h], 's-', c='#f59e0b', lw=1.5, ms=4, alpha=0.7)
        ax2b.set_ylabel('Entropy', color='#f59e0b')
        ax2.set_ylabel('τ', color='#8b5cf6')
        ax2.set_xlabel('Prompt')
        ax2.set_title('Entropy-Guided', fontweight='bold')
    
    plt.suptitle(f'{mname.upper()}: τ-Adaptation', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'tau_evolution_{mname}.png', dpi=150, bbox_inches='tight')
    plt.show()